# Team findings: a common comparison

We compare our wheel-wear and profile-shape findings using the same truck assessments and held-out cars. Acceleration and displacement remain separate outcomes. Our questions are whether the models outperform simple controls and whether added measurements help consistently across cars.

This is exploratory analysis of a small, repeatedly studied sample. A higher score does not establish a physical cause. We retain negative and mixed results.

Run each cell from a fresh kernel. Original data stay unchanged; tables and figures are saved under `outputs/team_comparison/`.

Follow [INSTALL.md](../INSTALL.md) for setup, figure instructions and outputs.


In [ ]:
# Shared input locations; see INSTALL.md for the Teams download layout.
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = ROOT
from team_data import load_benchmark_data
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
from xgboost import XGBClassifier

inputs = DATA
data = load_benchmark_data(inputs)
trucks = data["trucks"]
output = root / "outputs/team_comparison"
output.mkdir(parents=True, exist_ok=True)
display(pd.Series({"Cars": trucks.Car.nunique(), "Truck/rail assessments": len(trucks)}))


## Comparison rules

Each outer fold holds out one complete car, keeping its trucks and rail
assessments together. Logistic regression, RF and XGBoost use fixed settings;
the kernel-ridge comparisons retain their six-setting, three-inner-car-fold
tuning. Scaling uses training rows only. Rail categories are the seven supplied
conditions, so this tests new cars under existing rails.

The common feature block uses Dom's eight wheel measurements as averages and
absolute left/right differences, plus mean conicity and rail identity. Axle
features are averaged within each truck. Aaron's expanded RF shares several
of those definitions, retains its original inputs and uses a different full
list. These are overlapping feature ideas tested with different classifiers,
not independent discoveries or identical models.

We also compare Aaron's original and expanded RF lists under the same complete-
car folds. His personal notebook groups by truck, so its original scores answer
a different question. Shape/contact comparisons use matching controls. Balanced
accuracy, hunting recall and no-hunting recall are reported together.

Onset below 80 mph is positive; 80 means no onset recorded below the test limit.
These pipelines have different tuning budgets and were developed on this small
sample. Their scores do not establish an independently validated winner.

In [ ]:
rail = pd.get_dummies(trucks.Rail_Profile.astype(pd.CategoricalDtype(categories=range(1, 8))), prefix="rail", dtype=float)
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", C=1)),
    "Random forest": RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=2, learning_rate=.1, min_child_weight=3, random_state=42, n_jobs=1, eval_metric="logloss"),
}
experiments = [(name, "core", estimator) for name, estimator in models.items()]
experiments += [("RF original", "rf_original", models["Random forest"]),
                ("RF engineered", "rf_engineered", models["Random forest"]),
                ("LR: wear + conicity + rail", "wear", models["Logistic regression"]),
                ("LR: wear + shape + rail", "wear_shape", models["Logistic regression"])]
rows = []
for target in ["acceleration_hunted", "displacement_hunted"]:
    y = trucks[target].to_numpy(dtype=int)
    for name, features, estimator in experiments:
        X = np.column_stack([trucks[data["feature_sets"][features]].to_numpy(float), rail.to_numpy()])
        assert np.isfinite(X).all()
        for train, test in LeaveOneGroupOut().split(X, y, groups=trucks.Car):
            assert not set(trucks.Car.iloc[train]) & set(trucks.Car.iloc[test])
            fitted = clone(estimator).fit(X[train], y[train])
            predicted = fitted.predict(X[test]).astype(int)
            for index, prediction in zip(test, predicted):
                row = trucks.iloc[index]
                rows.append(dict(Car=int(row.Car), Truck=int(row.Truck), Rail_Profile=int(row.Rail_Profile), target=target, model=name, actual=int(y[index]), predicted=int(prediction)))
        print(target, name, "completed")
predictions = pd.DataFrame(rows)
assert not predictions.duplicated(["Car", "Truck", "Rail_Profile", "target", "model"]).any()
predictions.to_csv(output / "held_out_predictions.csv", index=False)


In [ ]:
from contact_hunting import evaluate_models
key = ["Car", "Truck", "Rail_Profile"]
kernel_features = {
    "KRR: conicity + rail": data["feature_sets"]["conicity"],
    "KRR: conicity + shape + rail": data["feature_sets"]["conicity_shape"],
    "KRR: wear + conicity + rail": data["feature_sets"]["wear"],
    "KRR: wear + shape + rail": data["feature_sets"]["wear_shape"],
    "KRR: wear + contact + rail": data["feature_sets"]["wear_contact"],
}
settings_frames = []
for target in ["acceleration_hunted", "displacement_hunted"]:
    _, held_out, settings = evaluate_models(trucks, target, feature_sets=kernel_features)
    assert not held_out.duplicated(key).any()
    assert held_out[key].equals(trucks[key])
    assert np.array_equal(held_out.actual, trucks[target])
    for model in ["Always hunting", "Rail only", *kernel_features]:
        frame = held_out[key + ["actual"]].copy()
        frame["target"], frame["model"], frame["predicted"] = target, model, held_out[model].to_numpy()
        predictions = pd.concat([predictions, frame], ignore_index=True)
    settings["target"] = target
    settings_frames.append(settings)
pd.concat(settings_frames, ignore_index=True).to_csv(output / "kernel_settings.csv", index=False)
assert not predictions.duplicated(key + ["target", "model"]).any()
predictions.to_csv(output / "held_out_predictions.csv", index=False)


## Results and per-car consistency

The table scores only held-out predictions. The error table keeps every car visible so a gain concentrated in one vehicle is easier to identify. These comparisons evaluate prespecified pipelines, including nested kernel tuning; they do not reproduce every teammate's original split or establish a winning algorithm.

In [ ]:
summary = []
for (target, model), group in predictions.groupby(["target", "model"]):
    tn, fp, fn, tp = confusion_matrix(group.actual, group.predicted, labels=[0, 1]).ravel()
    summary.append(dict(target=target, model=model, balanced_accuracy=balanced_accuracy_score(group.actual, group.predicted), hunting_recall=tp/(tp+fn), no_hunting_recall=tn/(tn+fp)))
metrics = pd.DataFrame(summary)
display(metrics.round(3))
metrics.to_csv(output / "metrics.csv", index=False)
errors = predictions.assign(error=lambda x: x.actual != x.predicted).groupby(["target", "Car", "model"]).error.sum().unstack()
display(errors)
errors.to_csv(output / "per_car_errors.csv")


## Paired checks and consolidated findings

We compare predictions for the same assessments and resample complete cars together. The ranges describe fixed held-out predictions, not retraining uncertainty or independent new-car validation. All planned comparisons are retained; these unadjusted exploratory ranges are not a multiple-testing significance claim.

In [ ]:
comparisons = [
    ("KRR: conicity + shape + rail", "KRR: conicity + rail", "Shape beyond conicity"),
    ("KRR: wear + shape + rail", "KRR: wear + conicity + rail", "Shape beyond standard wear"),
    ("KRR: wear + contact + rail", "KRR: wear + conicity + rail", "Contact beyond standard wear"),
    ("RF engineered", "RF original", "Aaron: engineered versus original"),
    ("Random forest", "Logistic regression", "Common features: RF versus logistic"),
    ("XGBoost", "Logistic regression", "Common features: XGBoost versus logistic"),
]
interval_rows = []
for target in predictions.target.unique():
    subset = predictions[predictions.target == target]
    for extended, baseline, question in comparisons:
        left = subset[subset.model == extended]
        right = subset[subset.model == baseline]
        joined = left.merge(right, on=key, validate="one_to_one", suffixes=("_new", "_base"))
        assert len(joined) == len(trucks)
        assert np.array_equal(joined.actual_new, joined.actual_base)
        cars = sorted(joined.Car.unique())
        counts = []
        for model_column in ["predicted_new", "predicted_base"]:
            counts.append(np.array([confusion_matrix(g.actual_new, g[model_column], labels=[0,1]).ravel()
                                    for _,g in joined.groupby("Car", sort=True)]))
        draws = np.random.default_rng(42).integers(0,len(cars),size=(2000,len(cars)))
        scores = []
        for c in counts:
            tn, fp, fn, tp = c[draws].sum(axis=1).T
            with np.errstate(divide="ignore",invalid="ignore"):
                scores.append(.5*(tn/(tn+fp)+tp/(tp+fn)))
        change = scores[0]-scores[1]
        change = change[np.isfinite(change)]
        assert len(change) > 0
        error_change = joined.assign(change=(joined.predicted_base != joined.actual_new).astype(int)
                                           -(joined.predicted_new != joined.actual_new).astype(int)).groupby("Car").change.sum()
        gain = balanced_accuracy_score(joined.actual_new, joined.predicted_new)-balanced_accuracy_score(joined.actual_new, joined.predicted_base)
        interval_rows.append(dict(target=target, question=question, gain_pp=100*gain,
            lower_pp=100*np.quantile(change,.025), upper_pp=100*np.quantile(change,.975),
            cars_improved=int((error_change>0).sum()),cars_worse=int((error_change<0).sum()),
            cars_tied=int((error_change==0).sum()),valid_resamples=len(change)))
intervals = pd.DataFrame(interval_rows)
display(intervals.round(2))
intervals.to_csv(output / "paired_comparisons.csv", index=False)


## Rail patterns and performance figures

The bars show held-car balanced accuracy. RF means random forest; LR means
logistic regression; KRR means kernel ridge. The two RF feature lists differ
from the common-feature RF. The dashed 0.5 line is the balanced-accuracy score
of an always-hunting predictor when both classes occur, not a significance test.

The rail chart is descriptive: the same trucks recur under seven conditions.
It does not isolate a rail-shape cause or test unseen rail profiles.

In [ ]:
rail_results = trucks.groupby("Rail_Profile").agg(
    acceleration_hunting_rate=("acceleration_hunted","mean"),
    displacement_hunting_rate=("displacement_hunted","mean"),
    mean_conicity=("conicity_mean","mean"))
rail_results = rail_results.join(data["rail_summary"].set_index("Rail_Profile"))
display(rail_results.round(3))
rail_results.to_csv(output / "rail_findings.csv")
fig, axes = plt.subplots(1,2,figsize=(13,6),sharey=True)
for ax,target,title in zip(axes, ["acceleration_hunted","displacement_hunted"], ["Acceleration","Displacement"]):
    view = metrics[metrics.target==target].set_index("model")["balanced_accuracy"].reindex(metrics.model.drop_duplicates())
    view.plot.barh(ax=ax,color="#257C89")
    ax.axvline(.5,color="gray",linestyle="--",linewidth=1)
    ax.set(title=title,xlabel="Held-out balanced accuracy",ylabel="",xlim=(0,1))
fig.tight_layout()
fig.savefig(output / "model_comparison.png",dpi=300,bbox_inches="tight")
fig.savefig(output / "model_comparison.pdf",bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(figsize=(10,5))
rail_results[["acceleration_hunting_rate","displacement_hunting_rate"]].rename(columns={"acceleration_hunting_rate": "Acceleration", "displacement_hunting_rate": "Displacement"}).plot.bar(ax=ax,rot=0)
ax.set(xlabel="Rail profile",ylabel="Fraction of truck assessments",title="Hunting rates by rail condition",ylim=(0,1))
fig.tight_layout();fig.savefig(output / "rail_hunting.png",dpi=300,bbox_inches="tight");fig.savefig(output / "rail_hunting.pdf",bbox_inches="tight");plt.show()


## What the results support

RF and XGBoost outperform logistic regression on the common wear features in
this run, most clearly for acceleration. Shape improves the displacement
comparison beyond mean conicity, but does not show a clear additional gain
after the standard-wear block. Contact additions also give mixed results.

The expanded RF changes direction between acceleration and displacement,
with paired ranges crossing zero. Shared average/difference features therefore
do not give one consistently better feature set. These results concern the
specified pipelines and 14 studied cars, not a general ranking of algorithms
or causal wheel characteristics.

Brandon's native FIT/LSTM score pools any-rail wheel labels and uses random
wheel splits, with augmentation before the second split. It can share related
profiles between train and test and answers a different question. That score
stays outside this common truck/rail ranking. A car-grouped split before
augmentation is needed for an unseen-car FIT evaluation.

In [ ]:
findings = intervals.copy()
findings["interpretation"] = np.select(
    [findings.lower_pp>0, findings.upper_pp<0],
    ["Positive exploratory change; inspect per-car consistency", "Negative exploratory change under this pipeline"],
    default="Direction uncertain in this sample")
display(findings[["target","question","gain_pp","lower_pp","upper_pp","cars_improved","cars_worse","interpretation"]].round(2))
findings.to_csv(output / "consolidated_findings.csv", index=False)
print("Completed shared evaluation; outputs saved locally.")
